# Azure LLM Inference Trace — Pruning Comparison

Compares the original trace (`AzureLLMInferenceTrace_conv.csv`) against the pruned trace
(`AzureLLMInferenceConvTrace_pruned_2048.csv`): how many requests were removed and what
fraction of the trace was cut.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ORIGINAL_CSV = "AzureLLMInferenceTrace_conv.csv"
PRUNED_CSV = "AzureLLMInferenceConvTrace_pruned_2048.csv"

orig = pd.read_csv(ORIGINAL_CSV)
pruned = pd.read_csv(PRUNED_CSV)
for df in (orig, pruned):
    df["TotalTokens"] = df["ContextTokens"] + df["GeneratedTokens"]

## Request count: before vs after

In [2]:
n_orig, n_pruned = len(orig), len(pruned)
n_removed = n_orig - n_pruned
pct_removed = 100 * n_removed / n_orig

summary = pd.DataFrame([
    {"Trace": "Original", "Requests": n_orig, "Share of original": "100.00%"},
    {"Trace": "Pruned (2048)", "Requests": n_pruned, "Share of original": f"{100 * n_pruned / n_orig:.2f}%"},
    {"Trace": "Removed", "Requests": n_removed, "Share of original": f"{pct_removed:.2f}%"},
])
display(summary)
display(Markdown(
    f"**{n_removed:,} requests removed out of {n_orig:,} "
    f"→ {pct_removed:.2f}% of the trace was cut "
    f"({n_pruned:,} requests remain, {100 * n_pruned / n_orig:.2f}%).**"
))

,Trace,Requests,Share of original
0,Original,19366,100.00%
1,Pruned (2048),16663,86.04%
2,Removed,2703,13.96%


**2,703 requests removed out of 19,366 → 13.96% of the trace was cut (16,663 requests remain, 86.04%).**

## Pruning rule verification

The pruned trace keeps exactly the requests with `ContextTokens < 2048`
(the pruned file's max ContextTokens is 2047; GeneratedTokens is unconstrained).

In [3]:
candidate = orig[orig["ContextTokens"] < 2048].reset_index(drop=True)
exact_match = candidate.equals(pruned.reset_index(drop=True))
print(f"rows with ContextTokens < 2048 in original : {len(candidate):,}")
print(f"rows in pruned trace                       : {n_pruned:,}")
print(f"row-by-row identical                       : {exact_match}")
assert exact_match, "pruned trace is not exactly orig[ContextTokens < 2048]"

rows with ContextTokens < 2048 in original : 16,663
rows in pruned trace                       : 16,663
row-by-row identical                       : True


## What was removed

In [4]:
removed = orig[orig["ContextTokens"] >= 2048]

def pct(part, whole):
    return f"{100 * part / whole:.2f}%"

token_summary = pd.DataFrame([
    {
        "Metric": "ContextTokens",
        "Original total": orig["ContextTokens"].sum(),
        "Removed total": removed["ContextTokens"].sum(),
        "Removed share": pct(removed["ContextTokens"].sum(), orig["ContextTokens"].sum()),
    },
    {
        "Metric": "GeneratedTokens",
        "Original total": orig["GeneratedTokens"].sum(),
        "Removed total": removed["GeneratedTokens"].sum(),
        "Removed share": pct(removed["GeneratedTokens"].sum(), orig["GeneratedTokens"].sum()),
    },
    {
        "Metric": "TotalTokens",
        "Original total": orig["TotalTokens"].sum(),
        "Removed total": removed["TotalTokens"].sum(),
        "Removed share": pct(removed["TotalTokens"].sum(), orig["TotalTokens"].sum()),
    },
])
display(token_summary)

display(Markdown("Removed requests — ContextTokens distribution:"))
display(removed["ContextTokens"].describe().to_frame().T.round(1))

,Metric,Original total,Removed total,Removed share
0,ContextTokens,22361870,9651260,43.16%
1,GeneratedTokens,4088665,216199,5.29%
2,TotalTokens,26450535,9867459,37.31%


Removed requests — ContextTokens distribution:

,count,mean,std,min,25%,50%,75%,max
ContextTokens,2703.0,3570.6,897.9,2050.0,2620.5,4077.0,4088.0,14050.0
